In [ ]:
import sys
sys.path.append("../..")
from src.training import helpers as tr_h
import scanpy as sc

adata = sc.read("/aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-08-14-01/data.h5ad")
adata = sc.read("/aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-08-14-01/data.h5ad")


In [ ]:
adata

In [80]:
ls ../../data

annotation_pancreas/  pp_data-24-09-24-02/  pp_data-25-04-29-05/
cellxgene/            pp_data-24-09-24-03/  pp_data-25-05-01-01/
ms/                   pp_data-24-09-25-01/  pp_data-25-05-01-02/
pp_data-24-08-26-01/  pp_data-24-09-25-02/  pp_data-25-05-01-03/
pp_data-24-09-02-01/  pp_data-24-09-25-03/  pp_data-25-05-01-04/
pp_data-24-09-04-01/  pp_data-24-10-06-01/  pp_data-25-05-01-05/
pp_data-24-09-05-01/  pp_data-24-11-07-01/  pp_data-25-05-07-01/
pp_data-24-09-09-01/  pp_data-24-11-10-01/  pp_data-25-05-07-02/
pp_data-24-09-09-02/  pp_data-24-11-11-01/  pp_data-25-08-12-01/
pp_data-24-09-13-01/  pp_data-24-11-12-01/  pp_data-25-08-12-02/
pp_data-24-09-14-01/  pp_data-24-11-13-01/  pp_data-25-08-14-01/
pp_data-24-09-15-01/  pp_data-25-04-29-01/  README.md
pp_data-24-09-22-01/  pp_data-25-04-29-02/  test_1/
pp_data-24-09-23-01/  pp_data-25-04-29-03/
pp_data-24-09-24-01/  pp_data-25-04-29-04/


In [93]:
adata.obs["sample_ids"] = [x.split(".")[1] for x in adata.obs["ids"]]
adata.obs.drop_duplicates(subset=["sample_ids"]).shape

(30150, 17)

In [ ]:
adata.obs["sample_ids"] = [x.split(".")[1] for x in adata.obs["ids"]]
adata.obs.drop_duplicates(subset=["sample_ids"]).shape

In [ ]:
# with controls - we see some samples can be in some cases controls others disease
adata.obs.groupby("sample_ids")["doid_id"].nunique().sort_values(ascending=False)

sample_ids
TCGA-QK-A6V9-01A    2
TCGA-QK-A6IJ-01A    2
TCGA-QK-A6II-01A    2
TCGA-QK-A6IH-01A    2
TCGA-QK-A6IG-01A    2
                   ..
GSM3529438          1
GSM3529437          1
GSM3529436          1
GSM3529435          1
GSM3529449          1
Name: doid_id, Length: 30150, dtype: int64

In [88]:
# only disease  - drop controls
adata.obs[adata.obs["doid_id"] != "Control"].groupby("sample_ids")["doid_id"].nunique().sort_values(ascending=False)

sample_ids
ERR6212579    2
ERR6212578    2
ERR6212577    2
ERR6212576    2
ERR6212575    2
             ..
GSM3636492    1
GSM3636491    1
GSM3636490    1
GSM3636489    1
GSM3636498    1
Name: doid_id, Length: 19283, dtype: int64

In [91]:
adata.obs.query("sample_ids == 'GSM5971892'")

,ids,dataset,dataset_id,batch,batch_id,dsaid,tissue,n_genes,disease,celltype,disease_study,library,doid_study,doid_id,do_id,doid_disease,sample_ids
17958,DSA05214.GSM5971892.Case,GSE199403,GSE199403,79,79,DSA05214,Lung,19344,Lung Disease,Lung Disease,Lung Disease,RNA-Seq,D,DOID:850,DOID:850,lung disease,GSM5971892
17987,DSA05215.GSM5971892.Case,GSE199403,GSE199403,79,79,DSA05215,Lung,19344,Lung Disease,Lung Disease,Lung Disease,RNA-Seq,D,DOID:850,DOID:850,lung disease,GSM5971892


In [21]:
results = list()
for n_samples in [1,2,3,4,5,6]:
    for n_datasets in [2,3,4,5,6]:
        adata_f = tr_h.clean_adata_qc(adata, n_samples=n_samples, n_dt=n_datasets)
        results.append({"thr_samples": n_samples, "thr_datasets": n_datasets, "n_datasets": adata_f.obs["dataset"].nunique(), "n_doids": adata_f.obs["doid_id"].nunique(), "n_samples": adata_f.shape[0], "n_samples_2": adata_f.obs["sample_ids"].nunique()})


Nº of datasets with +1 control samples: 660
Nº of datasets with +1 disease samples: 660
Nº of datasets with +1 samples (control and disease): 660
adata shape after filtering datasets with +1 samples: (35626, 20608)
Nº of passed diseases 131/ 131
Nº of passed dsaids 1159/ 1159
Nº of datasets with +1 control samples: 660
Nº of datasets with +1 disease samples: 660
Nº of datasets with +1 samples (control and disease): 660
adata shape after filtering datasets with +1 samples: (35626, 20608)
Nº of passed diseases 81/ 131
Nº of passed dsaids 987/ 1159
Nº of datasets with +1 control samples: 660
Nº of datasets with +1 disease samples: 660
Nº of datasets with +1 samples (control and disease): 660
adata shape after filtering datasets with +1 samples: (35626, 20608)
Nº of passed diseases 60/ 131
Nº of passed dsaids 892/ 1159
Nº of datasets with +1 control samples: 660
Nº of datasets with +1 disease samples: 660
Nº of datasets with +1 samples (control and disease): 660
adata shape after filtering

In [23]:
import pandas as pd
pd.DataFrame(results)

,thr_samples,thr_datasets,n_datasets,n_doids,n_samples,n_samples_2
0,1,2,660,132,35626,30150
1,1,3,575,82,28203,24648
2,1,4,518,61,24168,21795
3,1,5,453,44,20143,18559
4,1,6,414,36,18830,17310
5,2,2,660,132,35626,30150
6,2,3,575,82,28203,24648
7,2,4,518,61,24168,21795
8,2,5,453,44,20143,18559
9,2,6,414,36,18830,17310


In [71]:
import importlib
importlib.reload(tr_h)

<module 'src.training.helpers' from '/aloy/home/ddalton/projects/scGPT_playground/notebooks/exp/../../src/training/helpers.py'>

In [76]:
adata_f = tr_h.clean_adata_qc(adata, n_samples=2, n_dt=5)

# split
df_obs = adata_f.obs
test_obs = tr_h.split_stratified(
    df=df_obs,
    y_label="doid_id",  # should ALWAYS be on DOID! - OR  celltype be DOID! 
    group_label="dataset_id",
    split_size=10,
    seed=42,
)
tr_h.report_split(test_obs)


Nº of datasets with +2 control samples: 660
Nº of datasets with +2 disease samples: 660
Nº of datasets with +2 samples (control and disease): 660
adata shape after filtering datasets with +2 samples: (35626, 20608)
Nº of passed diseases 43/ 131
Nº of passed dsaids 780/ 1159
df shape: (20143, 17)
df diseases: (11546, 17)
All Labels: 44 ['Control', 'DOID:0050156', 'DOID:0080199', 'DOID:0081087', 'DOID:10286', 'DOID:10591', 'DOID:10652', 'DOID:11612', 'DOID:11714', 'DOID:11722', 'DOID:11725', 'DOID:11727', 'DOID:12377', 'DOID:12858', 'DOID:12930', 'DOID:13922', 'DOID:14250', 'DOID:14330', 'DOID:1485', 'DOID:1520', 'DOID:1909', 'DOID:2377', 'DOID:2394', 'DOID:2841', 'DOID:289', 'DOID:3068', 'DOID:3083', 'DOID:3310', 'DOID:3312', 'DOID:332', 'DOID:3459', 'DOID:3911', 'DOID:5015', 'DOID:5419', 'DOID:5844', 'DOID:6000', 'DOID:7148', 'DOID:8398', 'DOID:8577', 'DOID:8893', 'DOID:9074', 'DOID:9352', 'DOID:9744', 'DOID:9970']
Remove Controls - Labels: 43 ['DOID:0050156', 'DOID:0080199', 'DOID:008

In [77]:
test_obs[test_obs["test_split_1"] == 0]

,ids,dataset,dataset_id,batch,batch_id,dsaid,tissue,n_genes,disease,celltype,disease_study,library,doid_study,doid_id,do_id,doid_disease,sample_ids,test_split_1
50,DSA00030.GSM6634377.Control,GSE215424,GSE215424,461,461,DSA00030,Muscle,19402,Control,Control,Amyotrophic Lateral Sclerosis,RNA-Seq,D,Control,Control,Control,GSM6634377,0
51,DSA00030.GSM6634378.Control,GSE215424,GSE215424,461,461,DSA00030,Muscle,19402,Control,Control,Amyotrophic Lateral Sclerosis,RNA-Seq,D,Control,Control,Control,GSM6634378,0
52,DSA00030.GSM6634379.Control,GSE215424,GSE215424,461,461,DSA00030,Muscle,19402,Control,Control,Amyotrophic Lateral Sclerosis,RNA-Seq,D,Control,Control,Control,GSM6634379,0
53,DSA00030.GSM6634380.Control,GSE215424,GSE215424,461,461,DSA00030,Muscle,19402,Control,Control,Amyotrophic Lateral Sclerosis,RNA-Seq,D,Control,Control,Control,GSM6634380,0
54,DSA00030.GSM6634381.Control,GSE215424,GSE215424,461,461,DSA00030,Muscle,19402,Control,Control,Amyotrophic Lateral Sclerosis,RNA-Seq,D,Control,Control,Control,GSM6634381,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35593,DSA10102.GSM4990516.Control,GSE163908,GSE163908,212,212,DSA10102,nan,19402,Control,Control,Idiopathic Pulmonary Fibrosis,RNA-Seq,D,Control,Control,Control,GSM4990516,0
35594,DSA10102.GSM4990517.Control,GSE163908,GSE163908,212,212,DSA10102,nan,19402,Control,Control,Idiopathic Pulmonary Fibrosis,RNA-Seq,D,Control,Control,Control,GSM4990517,0
35595,DSA10102.GSM4990518.Case,GSE163908,GSE163908,212,212,DSA10102,nan,19402,Idiopathic Pulmonary Fibrosis,Idiopathic Pulmonary Fibrosis,Idiopathic Pulmonary Fibrosis,RNA-Seq,D,DOID:0050156,DOID:0050156,idiopathic pulmonary fibrosis,GSM4990518,0
35596,DSA10102.GSM4990519.Case,GSE163908,GSE163908,212,212,DSA10102,nan,19402,Idiopathic Pulmonary Fibrosis,Idiopathic Pulmonary Fibrosis,Idiopathic Pulmonary Fibrosis,RNA-Seq,D,DOID:0050156,DOID:0050156,idiopathic pulmonary fibrosis,GSM4990519,0


In [95]:

valid_obs = tr_h.split_stratified(
    df=test_obs[test_obs["test_split_1"]==0],
    y_label="doid_id",  # should ALWAYS be on DOID! - OR  celltype be DOID! 
    group_label="dataset_id",
    split_size=10,
    seed=42,
    new_label="valid_split_1"
)
tr_h.report_split(valid_obs, split_label="valid_split_1")
mask_train = valid_obs["valid_split_1"] == 0
mask_valid = valid_obs["valid_split_1"] == 1


df shape: (18768, 18)
df diseases: (10810, 18)
All Labels: 44 ['Control', 'DOID:0050156', 'DOID:0080199', 'DOID:0081087', 'DOID:10286', 'DOID:10591', 'DOID:10652', 'DOID:11612', 'DOID:11714', 'DOID:11722', 'DOID:11725', 'DOID:11727', 'DOID:12377', 'DOID:12858', 'DOID:12930', 'DOID:13922', 'DOID:14250', 'DOID:14330', 'DOID:1485', 'DOID:1520', 'DOID:1909', 'DOID:2377', 'DOID:2394', 'DOID:2841', 'DOID:289', 'DOID:3068', 'DOID:3083', 'DOID:3310', 'DOID:3312', 'DOID:332', 'DOID:3459', 'DOID:3911', 'DOID:5015', 'DOID:5419', 'DOID:5844', 'DOID:6000', 'DOID:7148', 'DOID:8398', 'DOID:8577', 'DOID:8893', 'DOID:9074', 'DOID:9352', 'DOID:9744', 'DOID:9970']
Remove Controls - Labels: 43 ['DOID:0050156', 'DOID:0080199', 'DOID:0081087', 'DOID:10286', 'DOID:10591', 'DOID:10652', 'DOID:11612', 'DOID:11714', 'DOID:11722', 'DOID:11725', 'DOID:11727', 'DOID:12377', 'DOID:12858', 'DOID:12930', 'DOID:13922', 'DOID:14250', 'DOID:14330', 'DOID:1485', 'DOID:1520', 'DOID:1909', 'DOID:2377', 'DOID:2394', 'DOID:2

In [103]:
adata.X[mask_train]

IndexError: boolean index did not match indexed array along dimension 0; dimension is 35626 but corresponding boolean dimension is 18768

In [101]:
import numpy as np
idxs_1 = np.argwhere(mask_train.to_numpy())

In [52]:
test_obs.columns

Index(['ids', 'dataset', 'dataset_id', 'batch', 'batch_id', 'dsaid', 'tissue',
       'n_genes', 'disease', 'celltype', 'disease_study', 'library',
       'doid_study', 'doid_id', 'do_id', 'doid_disease', 'sample_ids',
       'test_split_1'],
      dtype='object')

In [45]:
adata.obs[adata.obs["celltype"] == "Williams-Beuren Syndrome"]["doid_id"].unique()

['DOID:1928']
Categories (132, object): ['Control', 'DOID:235', 'DOID:289', 'DOID:332', ..., 'DOID:0060183', 'DOID:0060488', 'DOID:0080199', 'DOID:0081087']

In [104]:
from sklearn.model_selection import train_test_split
# Split to get indices only
train_idx, valid_idx = train_test_split(
    np.arange(len(df_obs)),
    test_size=0.1,
    shuffle=True,
    stratify=df_obs["doid_id"],
)
print(f"train_idx {train_idx.shape}, valid_idx {valid_idx.shape}")

train_idx (18128,), valid_idx (2015,)


In [111]:
list(train_idx)

[246,
 10159,
 1249,
 12865,
 9748,
 10876,
 8855,
 11557,
 11996,
 8990,
 3762,
 17744,
 15191,
 13069,
 16047,
 15980,
 11541,
 13237,
 9219,
 11554,
 611,
 14291,
 14348,
 3766,
 4175,
 4359,
 15029,
 5481,
 4858,
 9326,
 9317,
 12479,
 1229,
 10363,
 10322,
 6625,
 13245,
 18472,
 5617,
 18237,
 1184,
 16113,
 505,
 6899,
 4081,
 10647,
 13783,
 589,
 9884,
 11536,
 16486,
 16952,
 7523,
 12776,
 8478,
 398,
 2161,
 14253,
 6597,
 5196,
 17948,
 13511,
 13945,
 11847,
 17914,
 18942,
 15847,
 7197,
 857,
 9508,
 10949,
 12980,
 5951,
 8644,
 13823,
 4031,
 9752,
 20038,
 18137,
 10863,
 14578,
 855,
 15137,
 4695,
 6401,
 15743,
 2414,
 17876,
 4328,
 2731,
 16455,
 4974,
 9428,
 12262,
 3338,
 12255,
 14151,
 8936,
 10131,
 5932,
 1140,
 19645,
 20090,
 5673,
 1350,
 209,
 4016,
 5652,
 2240,
 2974,
 16987,
 18633,
 9162,
 10273,
 9842,
 15148,
 1467,
 14659,
 698,
 11007,
 7297,
 4855,
 4679,
 6940,
 15814,
 8029,
 4356,
 10871,
 392,
 7027,
 7450,
 2345,
 13356,
 11551,
 19220,


In [ ]:
from sklearn.model_selection import strai

In [105]:
train_idx

array([  246, 10159,  1249, ...,  1241,  1444,  3937])

In [46]:
adata.obs[adata.obs["doid_id"] == "DOID:1928"]["celltype"].unique()

['Williams Syndrome', 'Williams-Beuren Syndrome']
Categories (151, object): ['Acne Inversa', 'Acquired Immunodeficiency Syndrome', 'Acute Lymphoblastic Leukemia', 'Acute Myeloid Leukemia', ..., 'Type 2 Diabetes', 'Ulcerative Colitis', 'Williams Syndrome', 'Williams-Beuren Syndrome']

In [ ]:
adata.obs["celltype"]

In [107]:
idx_1 = np.argwhere(valid_obs["valid_split_1"].to_numpy() == 1)

In [109]:
idx_1.flatten()

array([  155,   156,   157, ..., 18618, 18619, 18620])

In [ ]:
rng = np.random.default_rng(42)
a = np.array([1, 2, 3, 4])
idx = rng.permutation(len(a))

In [112]:
(idx_1).shuffle()

AttributeError: 'numpy.ndarray' object has no attribute 'shuffle'